# 13 - Model Evaluation & Diagnostics

## Objective

Evaluate the selected Marketing Mix Model using statistical metrics and regression diagnostics.

This notebook answers:

- Is the model accurate?
- Are the regression assumptions reasonable?
- Are residuals random?
- Is there autocorrelation?
- Is heteroscedasticity present?
- Is the model suitable for business interpretation?


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error
)

import scipy.stats as stats
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan
import statsmodels.api as sm

ROOT = Path.cwd()
DATA = ROOT/"data"/"processed"/"marketing_mix_model_ready.csv"

df = pd.read_csv(DATA)

X=df.drop(columns=["Sales"])
y=df["Sales"]

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
)

model=Ridge(alpha=1.0)
model.fit(X_train,y_train)

pred=model.predict(X_test)
residuals=y_test-pred


## 1. Performance Metrics

In [ ]:

metrics=pd.DataFrame({
"Metric":[
"R²",
"Adjusted R²",
"MAE",
"RMSE",
"MAPE"
],
"Value":[
r2_score(y_test,pred),
1-(1-r2_score(y_test,pred))*((len(y_test)-1)/(len(y_test)-X_test.shape[1]-1)),
mean_absolute_error(y_test,pred),
np.sqrt(mean_squared_error(y_test,pred)),
mean_absolute_percentage_error(y_test,pred)
]
})

display(metrics)


## 2. Actual vs Predicted

In [ ]:

plt.figure(figsize=(6,6))
plt.scatter(y_test,pred)
plt.plot([y_test.min(),y_test.max()],[y_test.min(),y_test.max()],'r--')
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted")
plt.grid(True)
plt.show()


## 3. Residual Diagnostics

In [ ]:

fig=plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.hist(residuals,bins=25)
plt.title("Residual Distribution")

plt.subplot(1,2,2)
plt.scatter(pred,residuals)
plt.axhline(0,color="red")
plt.xlabel("Predicted")
plt.ylabel("Residual")
plt.title("Residual Plot")

plt.tight_layout()
plt.show()


## 4. QQ Plot

In [ ]:

stats.probplot(residuals, dist="norm", plot=plt)
plt.title("QQ Plot")
plt.show()


## 5. Normality Tests

In [ ]:

shapiro=stats.shapiro(residuals)
jb=stats.jarque_bera(residuals)

normality=pd.DataFrame({
"Test":["Shapiro-Wilk","Jarque-Bera"],
"Statistic":[shapiro.statistic,jb.statistic],
"P-Value":[shapiro.pvalue,jb.pvalue]
})

display(normality)


## 6. Durbin-Watson

In [ ]:

dw=durbin_watson(residuals)

print("Durbin-Watson:",round(dw,3))


## 7. Breusch-Pagan Test

In [ ]:

X_const=sm.add_constant(X_test)
bp=het_breuschpagan(residuals,X_const)

bp_df=pd.DataFrame({
"Metric":["LM Statistic","LM p-value","F Statistic","F p-value"],
"Value":bp
})

display(bp_df)


## 8. Regression Assumption Checklist

In [ ]:

checklist=pd.DataFrame({
"Assumption":[
"Linear relationship",
"Independent residuals",
"Homoscedasticity",
"Residual normality",
"Low multicollinearity (see VIF)"
],
"Status":[
"Review scatter plots",
"Check Durbin-Watson",
"Check Breusch-Pagan",
"Check QQ plot/Shapiro",
"Reviewed in Notebook 10"
]
})

display(checklist)


# Executive Summary

A model should not be selected based only on R².

Always evaluate:

- Predictive performance
- Stability
- Statistical assumptions
- Business interpretability

Only after passing these checks should the model be used for attribution and budget allocation.
